# Module 4: Search in Azure DocumentDB

**Time**: ~75 min  
**Environment**: Jupyter notebook in VS Code

This notebook is fully runnable. Enter your Azure DocumentDB connection string and OpenAI API key in Step 0, then run each cell in order. The notebook creates embeddings for the sample documents, stores them in Azure DocumentDB, creates vector and full-text indexes, and runs vector, BM25, fuzzy, phrase, and hybrid search.

> Full-text search in Azure DocumentDB is currently in gated preview. DiskANN vector search requires an M30 or higher cluster tier.


## Step 0: Connect and configure embeddings

This cell restores the MongoDB driver, accepts the DocumentDB connection string and OpenAI API key, and creates an HTTP helper for OpenAI embeddings.

In [ ]:
#r "nuget: MongoDB.Driver, 3.4.0"
using MongoDB.Bson;
using MongoDB.Driver;
using System.Linq;
using System.Net.Http.Headers;
using System.Text;
using System.Text.Json;

var connectionString = Environment.GetEnvironmentVariable("DOCUMENTDB_CONNECTION_STRING") ?? "<paste-your-azure-documentdb-connection-string-here>";
var openAiApiKey = Environment.GetEnvironmentVariable("OPENAI_API_KEY") ?? "<paste-your-openai-api-key-here>";
var embeddingModel = Environment.GetEnvironmentVariable("OPENAI_EMBEDDING_MODEL") ?? "text-embedding-3-small";
if (connectionString.Contains("<paste")) throw new Exception("Paste your Azure DocumentDB connection string in this cell or set DOCUMENTDB_CONNECTION_STRING.");
if (openAiApiKey.Contains("<paste")) throw new Exception("Paste your OpenAI API key in this cell or set OPENAI_API_KEY.");
var client = new MongoClient(connectionString);
var db = client.GetDatabase("docdbworkshop");
var collection = db.GetCollection<BsonDocument>("workshop_content");
var http = new HttpClient();
db.RunCommand<BsonDocument>(new BsonDocument("ping", 1))

## Step 1: Generate embeddings and load sample documents

This cell calls OpenAI for each document body, stores the embedding array, and loads the collection.

In [ ]:
async Task<BsonArray> CreateEmbeddingAsync(string text)
{
    using var request = new HttpRequestMessage(HttpMethod.Post, "https://api.openai.com/v1/embeddings");
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", openAiApiKey);
    request.Content = new StringContent(JsonSerializer.Serialize(new { model = embeddingModel, input = text }), Encoding.UTF8, "application/json");
    using var response = await http.SendAsync(request);
    var body = await response.Content.ReadAsStringAsync();
    response.EnsureSuccessStatusCode();
    using var json = JsonDocument.Parse(body);
    return new BsonArray(json.RootElement.GetProperty("data")[0].GetProperty("embedding").EnumerateArray().Select(v => v.GetDouble()));
}

var sourceDocs = new[] {
    new { Id="doc-search-001", Title="DiskANN vector indexing", Category="vector", Body="Azure DocumentDB supports DiskANN vector indexes for high recall semantic similarity search over embeddings stored with documents.", Sku="SEARCH-VEC-001" },
    new { Id="doc-search-002", Title="BM25 keyword search", Category="full-text", Body="Azure DocumentDB full-text search ranks keyword matches with BM25 and exposes scores through searchScore metadata.", Sku="SEARCH-FTS-001" },
    new { Id="doc-search-003", Title="Hybrid search with RRF", Category="hybrid", Body="Hybrid search combines BM25 keyword results with vector results and fuses the ranked lists using Reciprocal Rank Fusion.", Sku="SEARCH-HYB-001" },
    new { Id="doc-search-004", Title="RAG grounding", Category="rag", Body="Retrieval augmented generation retrieves relevant chunks from Azure DocumentDB and grounds the model answer in that context.", Sku="RAG-PIPE-001" }
};
collection.Database.DropCollection("workshop_content");
var docs = new List<BsonDocument>();
foreach (var item in sourceDocs)
{
    docs.Add(new BsonDocument { {"_id", item.Id}, {"title", item.Title}, {"category", item.Category}, {"body", item.Body}, {"sku", item.Sku}, {"embedding", await CreateEmbeddingAsync(item.Body)} });
}
collection = db.GetCollection<BsonDocument>("workshop_content");
collection.InsertMany(docs);
var embeddingDimensions = docs[0]["embedding"].AsBsonArray.Count;
new { Loaded = collection.CountDocuments(FilterDefinition<BsonDocument>.Empty), EmbeddingDimensions = embeddingDimensions }

## Step 2: Create vector and full-text indexes

The vector index uses the dimension count returned by the embedding model.

In [ ]:
db.RunCommand<BsonDocument>(new BsonDocument{{"createIndexes","workshop_content"},{"indexes",new BsonArray{new BsonDocument{{"name","idx_embedding_diskann"},{"key",new BsonDocument("embedding","cosmosSearch")},{"cosmosSearchOptions",new BsonDocument{{"kind","vector-diskann"},{"dimensions",embeddingDimensions},{"similarity","COS"},{"maxDegree",32},{"lBuild",64}}}}}}});
db.RunCommand<BsonDocument>(new BsonDocument{{"createSearchIndexes","workshop_content"},{"indexes",new BsonArray{new BsonDocument{{"name","idx_body_fts"},{"definition",new BsonDocument("mappings",new BsonDocument{{"dynamic",false},{"fields",new BsonDocument("body",new BsonDocument("type","string"))}})}}}}});

## Step 3: Generate a query embedding and run vector search

The same embedding model creates the query vector used by `$search.cosmosSearch`.

In [ ]:
var searchText = "semantic retrieval for RAG";
var queryVector = await CreateEmbeddingAsync(searchText);
collection.Aggregate<BsonDocument>(new[] {
    new BsonDocument("$search", new BsonDocument("cosmosSearch", new BsonDocument{{"path","embedding"},{"vector",queryVector},{"k",3}})),
    new BsonDocument("$project", new BsonDocument{{"_id",0},{"title",1},{"category",1},{"score",new BsonDocument("$meta","searchScore")}})
}).ToList()

## Step 4: Run BM25 full-text search

This query searches exact keywords and returns BM25 scores.

In [ ]:
collection.Aggregate<BsonDocument>(new[] {
    new BsonDocument("$search", new BsonDocument{{"index","idx_body_fts"},{"text",new BsonDocument{{"query","BM25 ranking"},{"path","body"}}}}),
    new BsonDocument("$limit", 5),
    new BsonDocument("$project", new BsonDocument{{"_id",0},{"title",1},{"score",new BsonDocument("$meta","searchScore")}})
}).ToList()

## Step 5: Run hybrid search

This cell embeds the query, runs BM25 and vector retrieval, and fuses result ranks with RRF.

In [ ]:
double RrfScore(int rank, int k = 60) => 1.0 / (k + rank + 1);
var userQuery = "semantic retrieval for RAG";
var hybridVector = await CreateEmbeddingAsync(userQuery);
var keywordHits = collection.Aggregate<BsonDocument>(new[] { new BsonDocument("$search", new BsonDocument{{"index","idx_body_fts"},{"text",new BsonDocument{{"query",userQuery},{"path","body"}}}}), new BsonDocument("$limit",5), new BsonDocument("$project",new BsonDocument{{"_id",1},{"title",1}}) }).ToList();
var vectorHits = collection.Aggregate<BsonDocument>(new[] { new BsonDocument("$search", new BsonDocument("cosmosSearch", new BsonDocument{{"path","embedding"},{"vector",hybridVector},{"k",5}})), new BsonDocument("$project",new BsonDocument{{"_id",1},{"title",1}}) }).ToList();
var scores = new Dictionary<string,double>(); var titles = new Dictionary<string,string>();
foreach (var list in new[] { keywordHits, vectorHits }) for (var i = 0; i < list.Count; i++) { var id = list[i]["_id"].ToString(); titles[id] = list[i]["title"].ToString(); scores[id] = scores.GetValueOrDefault(id) + RrfScore(i); }
scores.OrderByDescending(x => x.Value).Take(5).Select(x => new { Id = x.Key, Title = titles[x.Key], Score = x.Value }).ToList()